In [1]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
import wave

def check_wav_specs(file_path):
    with wave.open(file_path, 'rb') as wav_file:
        sample_rate = wav_file.getframerate()
        num_channels = wav_file.getnchannels()
        num_frames = wav_file.getnframes()

        print(f"Sample Rate: {sample_rate} Hz")
        print(f"Kanäle: {num_channels}")
        print(f"Frames: {num_frames}")
        print(f"Dauer: {num_frames / sample_rate:.2f} Sekunden")

check_wav_specs("../dataset/yes/00f0204f_nohash_0.wav")

Sample Rate: 16000 Hz
Kanäle: 1
Frames: 16000
Dauer: 1.00 Sekunden


In [3]:
# Check GPU
import torch

def check_gpu_status():
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_available}")

    if cuda_available:
        num_gpus = torch.cuda.device_count()
        print(f"Number of GPUs detected: {num_gpus}\n")
        for i in range(num_gpus):
            print(f"--- GPU {i} ---")
            print(f"Name: {torch.cuda.get_device_name(i)}")
            print(f"Compute Capability: {torch.cuda.get_device_capability(i)}")
            total_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            print(f"Total Memory: {total_memory:.2f} GB")
    else:
        print("PyTorch cannot detect a compatible GPU. It will default to CPU.")

check_gpu_status()

CUDA Available: True
Number of GPUs detected: 1

--- GPU 0 ---
Name: NVIDIA GeForce RTX 5070 Laptop GPU
Compute Capability: (12, 0)
Total Memory: 7.96 GB


In [4]:
# 1. Configuration
DATA_DIR = "../dataset/"
CLASSES = ["yes", "no", "up", "down"]
TARGET_SAMPLE_RATE = 16000
NUM_SAMPLES = 16000 # 1 second of audio
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# 2. Data Preparation
def get_files_and_labels():
    file_paths = []
    labels = []
    class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

    for cls_name in CLASSES:
        cls_dir = os.path.join(DATA_DIR, cls_name)
        if not os.path.exists(cls_dir):
            continue
        for file in os.listdir(cls_dir):
            if file.endswith(".wav"):
                file_paths.append(os.path.join(cls_dir, file))
                labels.append(class_to_idx[cls_name])

    return file_paths, labels


file_paths, labels = get_files_and_labels()
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

In [6]:
# 3. Dataset Class
class KeywordDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=TARGET_SAMPLE_RATE, n_mels=64
        )

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.paths[idx])

        # Resample if needed
        if sr != TARGET_SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, TARGET_SAMPLE_RATE)(waveform)

        # Mono conversion
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Pad or truncate to fixed length
        if waveform.shape[1] > NUM_SAMPLES:
            waveform = waveform[:, :NUM_SAMPLES]
        elif waveform.shape[1] < NUM_SAMPLES:
            waveform = F.pad(waveform, (0, NUM_SAMPLES - waveform.shape[1]))

        mel_spec = self.mel_transform(waveform)
        return mel_spec, torch.tensor(self.labels[idx], dtype=torch.long)

train_loader = DataLoader(KeywordDataset(train_paths, train_labels), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(KeywordDataset(test_paths, test_labels), batch_size=BATCH_SIZE, shuffle=False)

In [7]:

# 4. Model Architecture
class KeywordCNN(nn.Module):
    def __init__(self, num_classes):
        super(KeywordCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc1 = nn.Linear(32 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        logits = self.fc2(x)
        return logits
        # Note: PyTorch CrossEntropyLoss expects raw logits.
        # Softmax is applied in the predict function below.

    def predict(self, x):
        logits = self.forward(x)
        return F.softmax(logits, dim=1)

model = KeywordCNN(num_classes=len(CLASSES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# 5. Training Loop
print(f"Training on {DEVICE}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_acc = 100. * correct / total

    # Validation
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            test_total += targets.size(0)
            test_correct += predicted.eq(targets).sum().item()

    test_acc = 100. * test_correct / test_total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print("Training complete.")

Training on cuda...
Epoch 1/10 | Loss: 1.1844 | Train Acc: 52.36% | Test Acc: 62.01%
Epoch 2/10 | Loss: 0.8650 | Train Acc: 68.13% | Test Acc: 70.81%
Epoch 3/10 | Loss: 0.6896 | Train Acc: 75.08% | Test Acc: 77.61%
Epoch 4/10 | Loss: 0.5639 | Train Acc: 79.88% | Test Acc: 81.09%
